# SOLIDER Swin-Small partial fine-tune for Warehouse cameras (hardened)

Companion to `docs/superpowers/specs/2026-05-28-reid-fine-tune-design.md`.

**Goal:** push back-cam (0394 / 0395 / 0396) cross-camera ReID margin above the MCPT `sim_th = 0.85` gate.

**Strategy:** partial fine-tune (Swin stage index 3 + BN-neck + ID head). Stages 0-2 frozen. ID + hard-triplet + center loss. fp32 (no AMP, for stability). Leave scene_044 out for eval.

**Robustness (2026-05-31 hardening):** data is downloaded from the public, un-gated HuggingFace dataset and normalized in-notebook (no manual upload); crops use reliable sequential decode; training is fp32 with center folded into the main optimizer (no AMP/scaler-vs-center NaN); eval is guarded when the holdout is absent. Run cells top to bottom.

**Recommended runtime:** Colab Pro V100 / A100. Free T4 works (slower).

## Phase 1 — Environment & GPU check

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime required. Runtime > Change runtime type > GPU.'
print('device:', torch.cuda.get_device_name(0))
# Guard: torch must have CUDA kernels for THIS GPU's arch (Kaggle/Colab swap GPUs).
_cap = torch.cuda.get_device_capability(); _sm = f'sm_{_cap[0]}{_cap[1]}'
print('gpu arch:', _sm, '| torch arch_list:', torch.cuda.get_arch_list())
assert any(_sm in a for a in torch.cuda.get_arch_list()), (
    f'torch {torch.__version__} has NO kernels for this GPU ({_sm}). A pip install '
    'likely replaced torch. Kaggle: Factory reset (Run > Factory reset) and do NOT '
    'reinstall torch/torchvision; rerun. Do not upgrade torch.')

In [ ]:
import os, sys, subprocess
WORKDIR = '/content'; os.chdir(WORKDIR)
REPO = '/content/aic24-nvidia'
REPO_URL = 'https://github.com/harshbhatt-awl/aic24-nvidia.git'
REPO_BRANCH = 'main'   # SOLIDER backbone code lives on main

if not os.path.isdir(REPO):
    r = subprocess.run(['git','clone','--depth','1','-b',REPO_BRANCH,REPO_URL,REPO])
    if r.returncode != 0:
        raise RuntimeError(
            f'git clone failed (rc={r.returncode}). If the repo is private, set a token:\n'
            "  !git clone https://<USER>:<TOKEN>@github.com/harshbhatt-awl/aic24-nvidia.git")
os.chdir(REPO)
if REPO not in sys.path: sys.path.insert(0, REPO)
# Deps: install ONLY huggingface_hub + gdown (neither depends on torch). Do NOT add
# timm/einops/torch/torchvision — timm pulls a torch wheel whose CUDA kernels may not
# match the Kaggle/Colab GPU -> "CUDA error: no kernel image is available". The
# vendored SOLIDER needs only torch + cv2 + numpy, all pre-installed on Colab/Kaggle.
import torch as _tb; _vb = _tb.__version__
!pip install -q huggingface_hub gdown
print('cwd:', os.getcwd(), '| torch before install:', _vb,
      '(must be unchanged; if a later GPU op says "no kernel image", a pip install '
      'replaced torch -> Factory reset and do not reinstall torch)')

In [ ]:
# SOLIDER base checkpoint (Google Drive id in aic24_nvidia/models/solider/__init__.py).
import os, subprocess
os.makedirs('weights', exist_ok=True)
CKPT = 'weights/solider_swin_small.pth'
GDRIVE_ID = '1C-aIZdFyjFsZX4W4feG-Ex39RU2Qvu3b'

def _ok(p):
    return os.path.exists(p) and os.path.getsize(p) > 50_000_000   # ~190MB expected

if not _ok(CKPT):
    for attempt in range(3):
        subprocess.run(['gdown', GDRIVE_ID, '-O', CKPT])
        if _ok(CKPT):
            break
        print(f'  gdown attempt {attempt+1} produced no/short file; retrying...')
    if not _ok(CKPT):
        raise RuntimeError(
            'Could not fetch the SOLIDER base checkpoint via gdown (Drive rate-limit?).\n'
            'Manual fallback: download from https://github.com/tinyvision/SOLIDER-REID\n'
            f'and place at {CKPT} (expected ~190MB), then re-run this cell.')
print('checkpoint OK:', os.path.getsize(CKPT)//1_000_000, 'MB')

In [ ]:
# Smoke-test the SOLIDER wrapper before touching data.
from aic24_nvidia.models.solider import load_solider_swin_small, SOLIDER_SIZE
import torch
_m = load_solider_swin_small(CKPT).cuda()
with torch.no_grad():
    _f = _m(torch.randn(2, 3, *SOLIDER_SIZE).cuda())
assert tuple(_f.shape) == (2, 768), _f.shape
print('SOLIDER wrapper OK — feat shape', tuple(_f.shape))
del _m, _f; torch.cuda.empty_cache()

## Phase 2 — Data: automated HuggingFace download + normalize

`nvidia/PhysicalAI-SmartSpaces` is **public and un-gated** (no token needed). We download a few train scenes + the held-out eval scene, then normalize each raw scene (`camera_XXXX/video.mp4` + `*_2025_format.json`) into the flat `{videos/, calibration.json, ground_truth.json}` layout the cropper consumes. This replaces the old manual Drive-upload-then-normalize flow (the #1 source of "found 0 scenes").

Edit `TRAIN_SCENES` to add/remove scenes. More scenes → more unique IDs → better fine-tune (target 50–200 IDs).

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os

HF_REPO = 'nvidia/PhysicalAI-SmartSpaces'
HOLDOUT_SCENE = 'scene_044'                       # eval; never trained on
TRAIN_SCENES = ['scene_041', 'scene_042', 'scene_043']   # edit to taste
ALL_SCENES = TRAIN_SCENES + [HOLDOUT_SCENE]

RAW_ROOT = Path('/content/mtmc_raw')              # raw HF download
DATA_ROOT = Path('/content/data_norm')            # normalized flat layout
RAW_ROOT.mkdir(parents=True, exist_ok=True); DATA_ROOT.mkdir(parents=True, exist_ok=True)

for sc in ALL_SCENES:
    dst = RAW_ROOT / 'MTMC_Tracking_2024' / 'val' / sc
    if dst.exists() and list(dst.glob('camera_*/video.mp4')):
        print(f'[skip] {sc} already present'); continue
    print(f'[download] {sc} (~1.5-2GB)...')
    snapshot_download(HF_REPO, repo_type='dataset',
                      allow_patterns=[f'MTMC_Tracking_2024/val/{sc}/*'],
                      local_dir=str(RAW_ROOT))
print('raw scenes:', sorted(os.listdir(RAW_ROOT / 'MTMC_Tracking_2024' / 'val')))

In [ ]:
# Normalize raw -> flat layout (logic mirrors scripts/normalize_nvidia_scene.py, validated).
import json
from pathlib import Path

def _build_cameras(sensors):
    cams = {}
    for s in sensors:
        if s.get('type') != 'camera':
            continue
        ext = s['extrinsicMatrix']            # 3x4 [R|t]
        cams[s['id'].lower()] = {
            'K': s['intrinsicMatrix'],
            'R': [[r[0], r[1], r[2]] for r in ext],
            't': [ext[0][3], ext[1][3], ext[2][3]],
        }
    return cams

def _build_annotations(gt2025, cam_names):
    local_to_abs = {f'Camera_{i:04d}': cam_names[i] for i in range(len(cam_names))}
    anns = []
    for frame_str, objs in gt2025.items():
        fr = int(frame_str)
        for o in objs:
            if o.get('object_type') != 'Person':
                continue
            pid = int(o['object_id']); loc = o['3d_location']
            for cam_local, box in o.get('2d_bounding_box_visible', {}).items():
                ca = local_to_abs.get(cam_local)
                if ca is None:
                    continue
                x1, y1, x2, y2 = box
                anns.append({'camera': ca, 'frame': fr, 'person_id': pid,
                             'world_xy': [float(loc[0]), float(loc[1])],
                             'bbox_2d': [int(x1), int(y1), int(max(0, x2-x1)), int(max(0, y2-y1))]})
    return anns

def normalize_scene(src, out):
    src, out = Path(src), Path(out)
    out.mkdir(parents=True, exist_ok=True)
    if (out / 'ground_truth.json').exists() and (out / 'videos').exists():
        print(f'  [skip] {src.name} already normalized'); return
    calib = json.loads((src / 'calibration_2025_format.json').read_text())
    cams = _build_cameras(calib['sensors'])
    assert cams, f'no cameras parsed from {src}'
    (out / 'calibration.json').write_text(json.dumps({'cameras': cams}))
    (out / 'videos').mkdir(exist_ok=True)
    for cam in cams:
        sv = src / cam.lower() / 'video.mp4'
        if not sv.exists(): sv = src / cam / 'video.mp4'
        lt = out / 'videos' / f'{cam.lower()}.mp4'
        if lt.exists() or lt.is_symlink(): lt.unlink()
        if sv.exists(): lt.symlink_to(sv.resolve())
        else: print(f'    WARNING: missing video for {cam}')
    gt2025 = json.loads((src / 'ground_truth_2025_format.json').read_text())
    anns = _build_annotations(gt2025, list(cams.keys()))
    (out / 'ground_truth.json').write_text(json.dumps({'cameras': cams, 'annotations': anns}))
    del gt2025, anns                      # free the big GT structures
    print(f'  normalized {src.name}: {len(cams)} cams')

for sc in ALL_SCENES:
    split = 'val' if sc == HOLDOUT_SCENE else 'train'
    normalize_scene(RAW_ROOT / 'MTMC_Tracking_2024' / 'val' / sc, DATA_ROOT / split / sc)
print('normalized:', {s: (DATA_ROOT / ('val' if s == HOLDOUT_SCENE else 'train') / s / 'ground_truth.json').exists() for s in ALL_SCENES})

**Alternative data sources** (if you can't download from HF in your environment):
- **Drive:** mount Drive, point `RAW_ROOT` at a copy of `MTMC_Tracking_2024/`, and run the normalize cell (it handles the raw `*_2025_format.json` layout).
- **Pre-normalized:** if you already have the flat `{videos/, ground_truth.json, calibration.json}` layout, set `DATA_ROOT` to it and skip the two cells above.

## Phase 3 — Build crops + training manifest

Sequential frame decode (reliable; `cap.set` random-seek drops/duplicates frames on B-frame video). Bounded by `MAX_FRAME` and `FRAME_STEP` to keep decode time and disk in check. Each scene's GT is processed then freed. scene_044 is held out.

In [ ]:
from pathlib import Path
import json, csv, cv2, gc
from collections import defaultdict
from tqdm.auto import tqdm

CROPS_ROOT = Path('/content/crops'); CROPS_ROOT.mkdir(parents=True, exist_ok=True)
FRAME_STEP = 10        # sample every Nth frame
MAX_FRAME = 6000       # cap per camera (~first 200s @30fps) — bounds decode time/disk
BBOX_PAD = 8
MIN_BBOX_PX = 32
JPEG_Q = 92
MAX_CROPS_PER_CAM = 4000   # hard safety cap per (scene,camera)

def discover_scenes(root):
    out = []
    for split in ('train', 'val'):
        d = root / split
        if not d.exists(): continue
        for sc in sorted(d.iterdir()):
            if sc.is_dir() and (sc / 'ground_truth.json').exists():
                out.append((split, sc))
    return out

def parse_gt_by_cam_frame(gt_path, frame_step, max_frame):
    with gt_path.open() as f:
        gt = json.load(f)
    by_cam_frame = defaultdict(list)
    for ann in gt.get('annotations', []):
        bb = ann.get('bbox_2d') or ann.get('bbox')
        if not bb or len(bb) != 4: continue
        fr = int(ann['frame'])
        if fr > max_frame or fr % frame_step != 0: continue
        by_cam_frame[(ann['camera'], fr)].append((int(ann['person_id']), tuple(float(v) for v in bb)))
    del gt
    return by_cam_frame

def extract_crops(scene_dir, out_root):
    gt = scene_dir / 'ground_truth.json'
    if not gt.exists(): return []
    by_cf = parse_gt_by_cam_frame(gt, FRAME_STEP, MAX_FRAME)
    rows = []
    cams = sorted({c for c, _ in by_cf})
    for cam in cams:
        vid = scene_dir / 'videos' / f'{cam}.mp4'
        if not vid.exists():
            print(f'  missing video: {vid}'); continue
        try:
            cap = cv2.VideoCapture(str(vid))
            if not cap.isOpened():
                print(f'  cannot open: {vid}'); continue
            need = {fr for (c, fr) in by_cf if c == cam}
            last = max(need) if need else 0
            i, n_cam = 0, 0
            while i <= last and n_cam < MAX_CROPS_PER_CAM:
                ok, img = cap.read()
                if not ok: break
                if i in need:
                    H, W = img.shape[:2]
                    for pid, (x, y, w, h) in by_cf[(cam, i)]:
                        if w < MIN_BBOX_PX or h < MIN_BBOX_PX: continue
                        x0, y0 = max(0, int(x)-BBOX_PAD), max(0, int(y)-BBOX_PAD)
                        x1, y1 = min(W, int(x+w)+BBOX_PAD), min(H, int(y+h)+BBOX_PAD)
                        crop = img[y0:y1, x0:x1]
                        if crop.size == 0: continue
                        od = out_root / scene_dir.name / cam; od.mkdir(parents=True, exist_ok=True)
                        op = od / f'pid{pid:04d}_f{i:06d}.jpg'
                        if cv2.imwrite(str(op), crop, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_Q]):
                            rows.append((scene_dir.name, cam, i, pid, str(op))); n_cam += 1
                i += 1
            cap.release()
        except Exception as e:
            print(f'  ERROR on {cam}: {e!r} — skipping camera'); continue
    del by_cf; gc.collect()
    return rows

scenes = discover_scenes(DATA_ROOT)
print('scenes:', [(sp, s.name) for sp, s in scenes])
manifest_rows = []
for split, sc in tqdm(scenes, desc='scenes'):
    r = extract_crops(sc, CROPS_ROOT)
    print(f'  {sc.name}: {len(r)} crops')
    manifest_rows.extend(r)

MANIFEST = CROPS_ROOT / 'manifest.csv'
with MANIFEST.open('w', newline='') as f:
    w = csv.writer(f); w.writerow(['scene','camera','frame','person_id','crop_path']); w.writerows(manifest_rows)
print(f'\nTotal crops: {len(manifest_rows)} -> {MANIFEST}')

In [ ]:
import pandas as pd
df = pd.read_csv(MANIFEST)
df['split'] = df['scene'].apply(lambda s: 'val' if s == HOLDOUT_SCENE else 'train')
print(df.groupby(['split','scene']).agg(crops=('crop_path','size'), ids=('person_id','nunique'), cams=('camera','nunique')))
n_train = df[df.split=='train'].person_id.nunique(); n_val = df[df.split=='val'].person_id.nunique()
print(f'\nTrain unique IDs: {n_train} | Val unique IDs: {n_val}')
if n_train == 0:
    print('\n!!! TRAIN SPLIT EMPTY — add scenes to TRAIN_SCENES (Phase 2) and re-run Phases 2-3.')

## Phase 4 — Dataset / dataloader (identity-balanced PK sampler)

In [ ]:
import random, torch
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image
from collections import defaultdict
from aic24_nvidia.models.solider import SOLIDER_SIZE, SOLIDER_MEAN, SOLIDER_STD

TRAIN_TF = transforms.Compose([
    transforms.Resize(SOLIDER_SIZE), transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.1), transforms.ToTensor(),
    transforms.Normalize(SOLIDER_MEAN, SOLIDER_STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.4), value=0)])
EVAL_TF = transforms.Compose([
    transforms.Resize(SOLIDER_SIZE), transforms.ToTensor(),
    transforms.Normalize(SOLIDER_MEAN, SOLIDER_STD)])

class ReIDCropDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame['crop_path'].tolist(); self.cams = frame['camera'].tolist()
        uniq = sorted(set(frame['person_id'].tolist()))
        self.pid_to_idx = {p: i for i, p in enumerate(uniq)}
        self.labels = [self.pid_to_idx[p] for p in frame['person_id'].tolist()]
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (SOLIDER_SIZE[1], SOLIDER_SIZE[0]))   # tolerate a corrupt jpg
        return self.transform(img), self.labels[idx], self.cams[idx]

class PKSampler(Sampler):
    def __init__(self, labels, P=8, K=4, num_batches=None):
        self.P, self.K = P, K
        d = defaultdict(list)
        for i, y in enumerate(labels): d[y].append(i)
        self.label_to_idx = {y: ix for y, ix in d.items() if len(ix) >= K}
        self.labels = list(self.label_to_idx)
        self.num_batches = num_batches or max(1, len(labels) // (P * K))
    def __iter__(self):
        for _ in range(self.num_batches):
            chosen = random.sample(self.labels, self.P); batch = []
            for y in chosen: batch.extend(random.sample(self.label_to_idx[y], self.K))
            yield batch
    def __len__(self): return self.num_batches

train_df = df[df.split == 'train'].reset_index(drop=True)
val_df = df[df.split == 'val'].reset_index(drop=True)
P, K = 8, 4

if len(train_df) == 0:
    raise RuntimeError('train_df empty: only the held-out scene is present. Add scenes to TRAIN_SCENES (Phase 2).')
keepable = int((train_df.groupby('person_id').size() >= K).sum())
if keepable < P:
    raise RuntimeError(f'Only {keepable} train IDs have >=K={K} crops; need >=P={P}. '
                       'Lower FRAME_STEP / MAX_FRAME bounds in Phase 3, or add scenes.')
if len(val_df) == 0:
    print(f"WARNING: val empty (holdout '{HOLDOUT_SCENE}' not found). Eval is skipped; best is saved by train loss.")

train_ds = ReIDCropDataset(train_df, TRAIN_TF)
val_ds = ReIDCropDataset(val_df, EVAL_TF) if len(val_df) else None
sampler = PKSampler(train_ds.labels, P=P, K=K)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=P*K, shuffle=False, num_workers=2, pin_memory=True) if val_ds else None
NUM_TRAIN_IDS = len(set(train_ds.labels))
print(f'train crops={len(train_ds)} ids={NUM_TRAIN_IDS} keepable={keepable} | val crops={len(val_ds) if val_ds else 0}')
print(f'batches/epoch={len(sampler)} P={P} K={K}')

## Phase 5 — Model assembly + freeze (stage 3 + BN-neck + ID head)

In [ ]:
import torch.nn as nn
from aic24_nvidia.models.solider import load_solider_swin_small

class ReIDTrainer(nn.Module):
    def __init__(self, base_ckpt, num_ids):
        super().__init__()
        self.backbone = load_solider_swin_small(base_ckpt)
        self.id_head = nn.Linear(768, num_ids, bias=False)
        nn.init.normal_(self.id_head.weight, std=0.001)
    def forward(self, x):
        global_feat, _ = self.backbone.base(x)         # pre-BN (for triplet/center)
        bn_feat = self.backbone.bottleneck(global_feat)# post-BN (for id head; = inference feat)
        return self.id_head(bn_feat), global_feat, bn_feat
    @torch.no_grad()
    def embed(self, x):
        return self.backbone(x)                        # BN-neck, matches inference

model = ReIDTrainer(CKPT, NUM_TRAIN_IDS).cuda()
TRAINABLE = ('base.stages.3', 'base.norm3', 'bottleneck')
tr = fr = 0
for n, p in model.named_parameters():
    train = n.startswith('id_head') or any(k in n for k in TRAINABLE)
    p.requires_grad = train; tr += p.numel() if train else 0; fr += 0 if train else p.numel()
print(f'trainable={tr/1e6:.2f}M frozen={fr/1e6:.2f}M ratio={tr/(tr+fr):.1%}')
model.train()
_x, _y, _ = next(iter(train_loader)); _l, _gf, _bf = model(_x.cuda())
print('forward OK:', tuple(_l.shape), tuple(_gf.shape), tuple(_bf.shape)); del _x, _y, _l, _gf, _bf

## Phase 6 — Losses (ID + hard-triplet + center)

In [ ]:
import torch.nn.functional as F
def euclidean_dist_sq(x, y):
    return ((x*x).sum(1,keepdim=True) + (y*y).sum(1,keepdim=True).t() - 2*x@y.t()).clamp(min=1e-12)
class HardTripletLoss(nn.Module):
    def __init__(self, margin=0.3): super().__init__(); self.margin = margin
    def forward(self, feat, labels):
        d = euclidean_dist_sq(feat, feat).sqrt()
        same = labels.unsqueeze(0) == labels.unsqueeze(1)
        d_pos = (d * same.float()).max(1).values
        dn = d.clone(); dn[same] = float('inf'); d_neg = dn.min(1).values
        return F.relu(d_pos - d_neg + self.margin).mean()
class CenterLoss(nn.Module):
    def __init__(self, n, dim=768): super().__init__(); self.centers = nn.Parameter(torch.randn(n, dim)*0.01)
    def forward(self, feat, labels): return ((feat - self.centers[labels])**2).sum(1).mean()*0.5
ce = nn.CrossEntropyLoss(label_smoothing=0.1)
triplet = HardTripletLoss(0.3).cuda(); center = CenterLoss(NUM_TRAIN_IDS).cuda()
LAMBDA_ID, LAMBDA_TRI, LAMBDA_CEN = 1.0, 1.0, 5e-4
print('losses ready')

## Phase 7 — Training (fp32, center folded into main optimizer)

**Why fp32 + center-in-main-optimizer:** the old AMP path stepped the center optimizer with `GradScaler`-scaled gradients it didn't own → NaN/divergence. Folding center into the single AdamW (and dropping AMP) removes that failure mode entirely. Swin-Small at 384×128 fits comfortably in fp32 on a T4/V100/A100.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import time
from pathlib import Path

EPOCHS, WARMUP = 25, 5
bb = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('id_head')]
hd = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('id_head')]
opt = AdamW([{'params': bb, 'lr': 1e-5},
             {'params': hd, 'lr': 1e-4},
             {'params': center.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
sched = SequentialLR(opt, [LinearLR(opt, 0.01, total_iters=WARMUP),
                           CosineAnnealingLR(opt, T_max=max(1, EPOCHS-WARMUP))], milestones=[WARMUP])

CKPT_DIR = Path('weights'); CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH = CKPT_DIR / 'solider_swin_small_warehouse_ft_best.pth'
LAST_PATH = CKPT_DIR / 'solider_swin_small_warehouse_ft_last.pth'
def save_inference_ckpt(path): torch.save(model.backbone.state_dict(), path)  # base.* + bottleneck.* only

@torch.no_grad()
def embed_loader(loader):
    model.eval(); feats, labs, cams = [], [], []
    for x, y, cam in loader:
        f = model.embed(x.cuda(non_blocking=True))
        feats.append(F.normalize(f.float(), dim=1).cpu())
        labs.append(y if isinstance(y, torch.Tensor) else torch.tensor(y)); cams.extend(cam)
    return torch.cat(feats), torch.cat(labs), cams

def cross_cam_rank1_map(feats, labels, cams):
    import numpy as np
    feats, labels, cams = feats.numpy(), labels.numpy(), np.array(cams)
    sim = feats @ feats.T; np.fill_diagonal(sim, -np.inf)
    r1, aps = [], []
    for i in range(len(feats)):
        s = sim[i].copy(); s[cams == cams[i]] = -np.inf
        order = np.argsort(-s); order = order[s[order] > -np.inf]
        if len(order) == 0: continue
        rel = (labels[order] == labels[i]).astype('float32')
        if rel.sum() == 0: continue
        r1.append(rel[0]); cum = np.cumsum(rel)
        aps.append((cum/(np.arange(len(rel))+1)*rel).sum()/rel.sum())
    return (float(np.mean(r1)) if r1 else 0.0, float(np.mean(aps)) if aps else 0.0)

def quick_eval():
    if val_loader is None: return None, None
    f, l, c = embed_loader(val_loader); return cross_cam_rank1_map(f, l, c)

r1_0, map_0 = quick_eval()
print('pre-train cross-cam Rank-1 / mAP:', r1_0, map_0)

In [ ]:
best_metric = (r1_0 if r1_0 is not None else -1.0)
best_loss = float('inf'); history = []
for epoch in range(EPOCHS):
    model.train(); t0 = time.time()
    tot = {'total':0.,'id':0.,'tri':0.,'cen':0.}; correct = seen = 0
    for x, y, _ in train_loader:
        x = x.cuda(non_blocking=True)
        y = (y if isinstance(y, torch.Tensor) else torch.tensor(y)).cuda(non_blocking=True)
        opt.zero_grad(set_to_none=True)
        logits, gf, bf = model(x)
        l_id, l_tri, l_cen = ce(logits, y), triplet(gf, y), center(gf, y)
        loss = LAMBDA_ID*l_id + LAMBDA_TRI*l_tri + LAMBDA_CEN*l_cen
        if not torch.isfinite(loss):
            print(f'  WARNING: non-finite loss at ep{epoch+1}; skipping batch'); continue
        loss.backward(); opt.step()
        for k, v in zip(('total','id','tri','cen'), (loss, l_id, l_tri, l_cen)): tot[k] += float(v)
        correct += (logits.argmax(1) == y).sum().item(); seen += y.numel()
    sched.step()
    n = max(1, len(train_loader)); acc = correct/max(1, seen); avg_loss = tot['total']/n
    r1, mAP = quick_eval()
    msg = f"ep{epoch+1:02d}/{EPOCHS} loss={avg_loss:.3f} id={tot['id']/n:.3f} tri={tot['tri']/n:.3f} cen={tot['cen']/n:.4f} acc={acc:.3f}"
    if r1 is not None: msg += f' R1={r1:.4f} mAP={mAP:.4f}'
    msg += f' ({time.time()-t0:.1f}s)'; print(msg)
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'acc': acc, 'rank1': r1, 'mAP': mAP})
    save_inference_ckpt(LAST_PATH)
    improved = (r1 is not None and r1 > best_metric) or (r1 is None and avg_loss < best_loss)
    if improved:
        best_metric = r1 if r1 is not None else best_metric; best_loss = min(best_loss, avg_loss)
        save_inference_ckpt(BEST_PATH); print(f'  -> new best, saved {BEST_PATH.name}')
print(f'\nDone. best metric={best_metric:.4f} (pre-train R1={r1_0})')

## Phase 8 — Evaluation (guarded; skips cleanly if no holdout)

Cross-cam Rank-1/5/mAP, per-camera-pair cosine margin, and the critical back↔front same-ID cosine. Back/front camera sets are derived from the holdout scene's actual cameras.

In [ ]:
if val_loader is None:
    print('No holdout val set present — skipping Phase 8 eval. Train more scenes incl. scene_044 to enable it.')
else:
    import numpy as np, pandas as pd
    from aic24_nvidia.models.solider import load_solider_swin_small
    ft = load_solider_swin_small(str(BEST_PATH)).cuda(); ft.eval()
    F_all, L_all, C_all = [], [], []
    with torch.no_grad():
        for x, y, cam in val_loader:
            f = ft(x.cuda(non_blocking=True))
            F_all.append(F.normalize(f.float(), dim=1).cpu())
            L_all.append(y if isinstance(y, torch.Tensor) else torch.tensor(y)); C_all.extend(cam)
    Fn = torch.cat(F_all).numpy(); L = torch.cat(L_all).numpy(); C = np.array(C_all)
    sim = Fn @ Fn.T; np.fill_diagonal(sim, -np.inf)
    r1, r5, aps = [], [], []
    for i in range(len(Fn)):
        s = sim[i].copy(); s[C == C[i]] = -np.inf
        order = np.argsort(-s); order = order[s[order] > -np.inf]
        if len(order) == 0: continue
        rel = (L[order] == L[i]).astype('float32')
        if rel.sum() == 0: continue
        r1.append(rel[0]); r5.append(rel[:5].max())
        cum = np.cumsum(rel); aps.append((cum/(np.arange(len(rel))+1)*rel).sum()/rel.sum())
    print(f'Cross-cam Rank-1={np.mean(r1):.4f} Rank-5={np.mean(r5):.4f} mAP={np.mean(aps):.4f}')

    uniq = sorted(set(C_all)); nc = len(uniq)
    mean_same = np.full((nc, nc), np.nan); margin = np.full((nc, nc), np.nan)
    for a, ca in enumerate(uniq):
        ia = np.where(C == ca)[0]
        for b, cb in enumerate(uniq):
            ib = np.where(C == cb)[0]
            if not len(ia) or not len(ib): continue
            ssub = Fn[ia] @ Fn[ib].T; same = L[ia][:, None] == L[ib][None, :]
            if a == b: same = same & ~np.eye(len(ia), dtype=bool)
            if same.sum() == 0 or (~same).sum() == 0: continue
            mean_same[a, b] = ssub[same].mean(); margin[a, b] = ssub[same].mean() - ssub[~same].mean()
    print('\nSame-ID mean cosine:\n', pd.DataFrame(mean_same, index=uniq, columns=uniq).round(3))
    # Back/front derived from holdout cameras: highest-numbered 3 are the back cams here.
    BACK = set(uniq[-3:]); FRONT = set(uniq[:-3])
    bi = [i for i, c in enumerate(uniq) if c in BACK]; fi = [i for i, c in enumerate(uniq) if c in FRONT]
    bf_same = float(np.nanmean(mean_same[np.ix_(bi, fi)])); bf_margin = float(np.nanmean(margin[np.ix_(bi, fi)]))
    print(f'\nBack({sorted(BACK)}) <-> Front same-ID cosine = {bf_same:.4f} (target >0.50)')
    print(f'Back <-> Front margin = {bf_margin:.4f} (target >0.40)')

## Phase 9 — Save outputs (Drive optional)

In [ ]:
import json, shutil, os
SAVE_DIR = '/content/aic24_reid_finetune'
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/aic24_reid_finetune'
except Exception as e:
    print(f'Drive not available ({e!r}); saving locally to {SAVE_DIR}. Download manually.')
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy(BEST_PATH, f'{SAVE_DIR}/solider_swin_small_warehouse_ft.pth')
shutil.copy(LAST_PATH, f'{SAVE_DIR}/solider_swin_small_warehouse_ft_last.pth')
with open(f'{SAVE_DIR}/training_summary.json', 'w') as f:
    json.dump({'pre_train_rank1': r1_0, 'best_metric': best_metric, 'history': history,
               'epochs': EPOCHS, 'P_K': [P, K], 'train_scenes': TRAIN_SCENES,
               'lambda_id_tri_cen': [LAMBDA_ID, LAMBDA_TRI, LAMBDA_CEN]}, f, indent=2)
print('saved to', SAVE_DIR); print(os.listdir(SAVE_DIR))

## Phase 10 — Local integration (back on your machine)

1. Copy `solider_swin_small_warehouse_ft.pth` into `weights/`.
2. Add the `reid.checkpoint_path` knob (spec §Local integration): `ReidCfg.checkpoint_path` in `config.py`, thread it to `load_solider_swin_small` in `reid_solider.py`, set it in `configs/baseline.yaml`.
3. Rebuild reid→evaluate and compare:
   ```bash
   rm -rf outputs/baseline/{reid,sct,mct,evaluate}
   python pipeline.py all --config configs/baseline.yaml --run-id baseline
   ```
4. Decision rule (spec): back-cam global IDs up AND world HOTA ≥ +0.01 → ship as new baseline. The fine-tune targets **AssA** (the current v3.3 ceiling, 0.6423); it does not change the dropped count.
5. Commit: `git add notebooks/reid_finetune.ipynb && git commit -m 'feat(reid): hardened fine-tune notebook'`